## Importing libraries

In [1]:
import numpy as np
import pandas as pd
import re

## Importing data

Columns to drop

In [2]:
columns_to_not_use = [
# Same values for all rows
    'record_type' ,
    'record_subclass' ,
    'confirmation_location' ,
    'dia_bp' ,
    'group_number' ,
    'manufacturer_code' ,
    'patient_age_units' ,
    'patient_height' ,
    'patient_weight' ,
    'reason_for_procedure' ,
    'sys_bp' ,
    'text_size' ,
    'pp_interval' ,

# No value at all
    'assisting_provider' ,
    'attending_phy' ,
    'patient_middle_name' ,
    'patient_secondary_id' ,
    'patient_ssn' ,
    'procedural_diagnosis' ,
    'referral_source' ,
    'referring_phy' ,
]

Columns to format in a particular type

In [3]:
sure_string_cols = [
    'pseudopaz'
]
dtype = {col: 'string' for col in sure_string_cols}

string_cat_cols = [
    'site_number', # ???
    'record_status',
    'medications',
    'patient_height_units' ,
    'patient_location' ,
    'patient_race' ,
    'patient_race_str' ,
    'patient_room' ,
    'patient_sex' ,
    'patient_weight_units' ,
    'transmit_device_number' ,
    'reason_for_change' ,
]
dtype.update({col: 'string' for col in string_cat_cols})

Import the excel file

In [4]:
df = pd.read_excel('../raw_data/dati_escribe.xlsx',
    usecols=lambda col: col not in columns_to_not_use,
    dtype_backend='numpy_nullable', # For allowing the `string` dtype
    dtype=dtype
)

# Make the index start from 2 (as the original excel)
df.index = df.index + 2

## Parsing and reformatting

Defining `format_phy_name` function

In [5]:
def format_phy_name(value):
    """
    Requires pandas.
    TODO
    """
    # Handle NA values
    if pd.isna(value):
        return pd.NA

    # Count whitespaces
    dot_count = value.count('.')

    # Handle the only name with one dot
    if dot_count == 1:
        return value

    elif dot_count == 2:
        second_dot = value.rfind('.')
        name = value[second_dot -1 : second_dot]
        surname = value[value.rfind('.') +1 : ].strip()
        formatted_name = 'DR-' + name + '_' + surname
        
        return formatted_name

### Formatting `overreading_phy` column

In [6]:
# Only name with one dot
df['overreading_phy'] = df['overreading_phy'].replace('DR. GP BROSO', 'DR-GP_BROSO')

# Run the function
df['overreading_phy'] = df['overreading_phy'].apply(format_phy_name).astype('string')

### Splitting and formatting `interp_text` column

In [7]:
def split_column(value):

    # If row is already a NA value, leave unchanged
    if pd.isna(value):
        return pd.Series([pd.NA, pd.NA, pd.NA])

    # Split on the first ~~ from right
    parts = value.rsplit('~~', maxsplit=1)

    # Place a NA in the missing splitted parts
    for i, part in enumerate(parts):
        if part == '':
            parts[i] = pd.NA

    # Parse "parts[1]"
    if parts[1].startswith('REPORT NON CONFERM'):
        doc, datetime = 'RNC', pd.NaT
    elif parts[1].startswith('Refertato da _________________'):
        doc, datetime = 'phy_NA', pd.NaT
    elif parts[1].startswith('Refertato da '):
        doc_pro, date, hour = (re.sub(' +', ' ', parts[1])).strip().rsplit(' ', maxsplit=2)
        doc = format_phy_name(doc_pro[13:])
        datetime = pd.to_datetime('{} {}'.format(date, hour), format='%d/%m/%Y %H.%M.%S')

    if len(parts) == 2:
        return pd.Series([parts[0], doc, datetime])
    else:
        raise Exception('Abnormal row')

Pre-format row 1685, the only problematic

In [8]:
original = df.at[1685, 'interp_text']
df.at[1685, 'interp_text'] = '~' + original

In [9]:
# Get position and split
original_col_name = 'interp_text'
original_position = df.columns.get_loc(original_col_name)

# Split the column
split_df = df[original_col_name].apply(split_column)
split_df.columns = ['diagnosis', 'phy', 'phy_date']
split_df[['diagnosis', 'phy']] = split_df[['diagnosis', 'phy']].astype('string')

# Reconstruct dataframe: columns before + new columns + columns after
df = pd.concat([
    df.iloc[:, :original_position
        # +1 # Decomment for keeping the original column
    ],
    split_df,
    df.iloc[:, original_position+1:]
], axis=1)

### Merging similar values in columns

#### `comment_field`

Values are `{..., 'Dolore toracico': 8,'Dolore torace': 2, ...}` - `Dolore torace` values are merged to `Dolore toracico`

In [10]:
df['comment_field'] = df['comment_field'].replace('Dolore torace', 'Dolore toracico')

#### `note1`

Merging:
- `DOL. TORACICO` and `DOL TOR` and `DOLORETORAC` and `DOLORE TORACICO`
    - to `TORACALGIA`
- `ADDOMINALG` and `DOLORE ADDOMIN`
    - to `ADDOMINALGIA`
- `Dispnea`
    - to `DISPNEA`

In [11]:
df['note1'] = df['note1'].replace(['DOLORE TORACICO', 'DOLORETORAC', 'DOL. TORACICO', 'DOL TOR'], 'TORACALGIA')
df['note1'] = df['note1'].replace(['ADDOMINALG', 'DOLORE ADDOMIN'], 'ADDOMINALGIA')
df['note1'] = df['note1'].replace(['Dispnea'], 'DISPNEA')

#### `note2` (tanti auguri)

Merging:
- `.` and `.`
    - to `.`

In [12]:
# df[''] = df[''].replace([''], '')

#### `patient_location`

Merging:
- `OTO`
    - to `Oto`

In [13]:
df['patient_location'] = df['patient_location'].replace(['OTO'], 'Oto')

#### ``

Merging:
- `.` and `.`
    - to `.`

In [14]:
# df[''] = df[''].replace([''], '')

### Formatting `patient_date_of_birth` column

Since 33 rows have the date formatted as `06/17/1950` (for example row 790), `format='mixed'` is required

In [15]:
df['patient_date_of_birth'] = pd.to_datetime(df['patient_date_of_birth'], errors='raise', dayfirst=True, format='mixed')

## Final checks

In [16]:
df

,pseudopaz,site_number,record_status,acq_time,confirm_time,transmission_time,comment_field,medications,note1,note2,...,qrs_offset,t_offset,p_axis,qrs_axis,t_axis,v_rate,pr_interval,qrs_duration,qt_interval,qtc_interval
2,23311141116542459234782,22,2,2025-03-18 13:32:59,2025-03-18 14:00:02,2025-03-18 13:31:04,<NA>,||,<NA>,AMB CARDIO,...,539,771,20,14,57,111,214,92,324,390
3,23311141116542459234782,22,0,2025-03-18 13:32:59,NaT,2025-03-18 13:31:04,<NA>,||,<NA>,AMB CARDIO,...,539,771,20,14,57,111,214,92,324,390
4,44761223044751751230474,3,2,2023-10-06 13:08:14,2023-10-06 13:52:00,2023-10-06 13:19:15,<NA>,||,<NA>,BASE HOLTER,...,567,863,999,-7,19,67,0,108,404,420
5,44761223044751751230474,3,0,2023-10-06 13:08:14,NaT,2023-10-06 13:19:15,<NA>,||,<NA>,BASE HOLTER,...,567,863,999,-7,19,67,0,108,404,420
6,41430443687272900939689,3,2,2025-04-23 11:40:07,2025-04-23 15:12:30,2025-04-23 11:30:47,<NA>,||,<NA>,BIOTRONIK,...,551,858,36,-25,119,69,164,106,413,431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1784,8688775490799534962197,2,0,2019-06-11 18:41:55,NaT,2019-06-11 18:44:20,<NA>,||,DIFF RESP,<NA>,...,538,757,999,7,81,126,0,85,304,378
1785,8688775490799534962197,2,2,2021-08-04 14:48:54,2021-08-04 15:01:39,2021-08-04 14:45:08,<NA>,||,<NA>,<NA>,...,538,804,999,-35,75,107,0,90,356,418
1786,8688775490799534962197,2,2,2024-03-08 17:12:58,2024-03-08 17:38:18,2024-03-08 17:15:25,<NA>,||,<NA>,<NA>,...,554,856,999,-55,91,90,0,112,414,462
1787,8688775490799534962197,2,0,2024-03-08 17:12:58,NaT,2024-03-08 17:15:25,<NA>,||,<NA>,<NA>,...,554,856,999,-55,91,90,0,112,414,462


In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1787 entries, 2 to 1788
Data columns (total 41 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   pseudopaz               1787 non-null   string        
 1   site_number             1787 non-null   string        
 2   record_status           1787 non-null   string        
 3   acq_time                1787 non-null   datetime64[us]
 4   confirm_time            886 non-null    datetime64[us]
 5   transmission_time       1787 non-null   datetime64[us]
 6   comment_field           14 non-null     string        
 7   medications             1787 non-null   string        
 8   note1                   144 non-null    string        
 9   note2                   1047 non-null   string        
 10  overreading_phy         873 non-null    string        
 11  patient_age             1787 non-null   Int64         
 12  patient_date_of_birth   1787 non-null   datetime64[us]
 13 

## Export

In [18]:
df.to_parquet("../raw_data/parquets/dati_escribe.parquet"
    # , compression="gzip"
)

### Inconsistency between `pseudopaz` and other columns

#### Proof of the inconsistency

`patient_age`, `patient_date_of_birth`, `patient_race`, `patient_sex` are inconsistent

In [19]:
(df.groupby('pseudopaz')['patient_age'].nunique() == 1).all()
(df.groupby('pseudopaz')['patient_date_of_birth'].nunique() == 1).all()
(df.groupby('pseudopaz')['patient_race'].nunique() == 1).all()
(df.groupby('pseudopaz')['patient_sex'].nunique() == 1).all()

np.False_

#### See which `pseudopaz` have multiple characteristics that should be unique

In [21]:
incons = df.groupby('pseudopaz')['patient_sex'].nunique()
incons = incons[incons > 1].sort_values(ascending=False)
incons

pseudopaz
33794016081826308645769    3
10215810359558408931496    2
12363054132398258315171    2
13280116591091443765758    2
13346969065256785955969    2
                          ..
69796904191818555065251    2
71471812250303076605610    2
71531525468081571734508    2
7394299732913398640877     2
8688775490799534962197     2
Name: patient_sex, Length: 78, dtype: int64